# 1.0 Importando pacotes

In [1]:
import geopandas as gpd
from pathlib import Path
import pandas as pd
import numpy as np

# 2.0 Definindo caminhos

In [2]:
# Caminho da pasta de inputs
inputs_path = Path('./inputs')

# Caminho da pasta de outputs
outputs_path = Path('./outputs')

# Caminhos dos subsets
co_path = inputs_path / 'subset_co.parquet'
no2_path = inputs_path / 'subset_no2.parquet'
o3_path = inputs_path / 'subset_o3.parquet'
pm_path = inputs_path / 'subset_pm.parquet'
so2_path = inputs_path / 'subset_so2.parquet'

# 3.0 Carregando subsets

In [3]:
subset_co = gpd.read_parquet(co_path)
subset_no2 = gpd.read_parquet(no2_path)
subset_o3 = gpd.read_parquet(o3_path)
subset_pm = gpd.read_parquet(pm_path)
subset_so2 = gpd.read_parquet(so2_path)

# 4.0 Verificando validade de cada classe para cada estação
Verificando se cada estação possui ao menos uma via de ao menos uma faixa de ADT que se encontre dentro dos limites de cada classe de representatividade

In [4]:
# Função auxiliar para verificar se existem colunas de cada escala
def safe_any(df, like_str):
    """
    Filtra todas as colunas de um dataframe com uma substring comum no nome
    e verifica se há pelo menos um valor True em cada linha, retornando uma série de True.
    Caso contrário ou se estiver vazio, retorna uma série de False.
    
    Essa função é necessária porque, se não houver colunas no dataframe filtrado,
    a função .all retorna uma série de True, o que é enganoso.
    
    Parâmetros
    ----------
    df : dataframe 
        DataFrame com várias colunas contendo um elemento repetido no nome.
    like_str: str
        Substring comum nos nomes das colunas do DataFrame.

    Retorna
    -------
    filtered : série de booleans
        Série booleana com o mesmo tamanho que o DataFrame original.
    """
    # Filtra todas as colunas que contêm a string específica no nome
    filtered = df.filter(like=like_str)
    
    # Se não houver colunas correspondentes, retorna uma série de False
    if filtered.shape[1] == 0:
        return pd.Series([False] * len(df), index=df.index)
        
    return filtered.any(axis='columns')


# Aplicando a função para verificar a validade de cada rep_{classe} --------------

# CO ---------------------------------
subset_co['rep_micro_any'] = safe_any(subset_co, 'rep_micro')
subset_co['rep_bairro_any'] = safe_any(subset_co, 'rep_bairro')

# NO2 -------------------------------
subset_no2['rep_micro_any'] = safe_any(subset_no2, 'rep_micro')
subset_no2['rep_bairro_any'] = safe_any(subset_no2, 'rep_bairro')
subset_no2['rep_urb_any'] = safe_any(subset_no2, 'rep_urb')

# O3 --------------------------------
subset_o3['rep_bairro_any'] = safe_any(subset_o3, 'rep_bairro')
subset_o3['rep_urb_any'] = safe_any(subset_o3, 'rep_urb')

# PM -------------------------------
subset_pm['rep_meso_any'] = safe_any(subset_pm, 'rep_meso')
subset_pm['rep_bairro_any'] = safe_any(subset_pm, 'rep_bairro')
subset_pm['rep_urb_any'] = safe_any(subset_pm, 'rep_urb')

# SO2 -------------------------------------------
subset_so2['rep_micro_any'] = safe_any(subset_so2, 'rep_micro')
subset_so2['rep_bairro_any'] = safe_any(subset_so2, 'rep_bairro')


In [5]:
# subset_pm.loc[subset_pm.ID_OEMA.str.contains('Fercal Boa Vista')].iloc[[1]][[col for col in list(subset_pm.columns) if 'any' in col]]

In [8]:
subset_pm.filter(like='rep_bairro')

,rep_bairro_1k,rep_bairro_15k,rep_bairro_20k,rep_bairro_30k,rep_bairro_40k,rep_bairro_50k,rep_bairro_60k,rep_bairro_70k,rep_bairro_80k,rep_bairro_any
0,False,False,False,False,False,False,False,True,True,True
1,False,False,False,False,False,False,False,True,True,True
2,False,False,False,False,False,False,False,True,True,True
3,False,False,False,False,False,False,False,True,True,True
4,False,False,False,False,False,False,False,True,True,True
...,...,...,...,...,...,...,...,...,...,...
838,False,True,False,False,False,False,False,True,True,True
839,True,False,False,False,False,False,False,True,True,True
840,False,False,False,False,False,False,False,True,True,True
841,True,False,False,False,False,False,False,True,True,True


# 5.0 Salvando outputs

In [7]:
subset_co.to_parquet(outputs_path / 'subset_co.parquet')
subset_no2.to_parquet(outputs_path / 'subset_no2.parquet')
subset_o3.to_parquet(outputs_path / 'subset_o3.parquet')
subset_pm.to_parquet(outputs_path / 'subset_pm.parquet')
subset_so2.to_parquet(outputs_path / 'subset_so2.parquet')